In [4]:
import pandas as pd
import numpy as np

In [5]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder 

In [6]:
df = pd.read_csv('covid_toy.csv')

In [7]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [8]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [9]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [10]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

### 10 indexes do not have fever value or missing value so first need to do preprocessing 

### for yes or no use label encoder 

In [11]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.2)

In [12]:
X_train.head()

,age,gender,fever,cough,city
82,24,Male,98.0,Mild,Kolkata
35,82,Female,102.0,Strong,Bangalore
21,73,Male,98.0,Mild,Bangalore
80,14,Female,99.0,Mild,Mumbai
98,5,Female,98.0,Strong,Mumbai


# 1) Aam Zindagi ( If we do not know what is column transformer )

In [13]:
# adding simple imputer to fever column 
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data 
X_test_fever = si.fit_transform(X_test[['fever']])

X_train_fever.shape # all missing values get replaced by mean value 

(80, 1)

In [14]:
# Ordinal encoding -> cough 
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data 
X_test_cough = oe.fit_transform(X_test[['cough']])
X_test_cough , X_train_cough # -> both are in numpy array 

(array([[0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [0.]]),
 array([[0.],
        [1.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [1.],
    

In [15]:
# One hot encoding -> gender , city 
ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data 
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_test_gender_city.shape

(20, 4)

In [16]:
# Extracting age 
X_train_Age = X_train.drop(columns=['gender','fever','cough','city'])

X_test_age = X_test.drop(columns=['gender','fever','cough','city'])

X_train_Age.shape

(80, 1)

In [17]:
X_train_tranformed = np.concatenate((X_train_Age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)

# also the test data 
X_test_tranformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

In [18]:
X_train_tranformed

array([[ 24.        ,  98.        ,   1.        ,   0.        ,
          1.        ,   0.        ,   0.        ],
       [ 82.        , 102.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   1.        ],
       [ 73.        ,  98.        ,   1.        ,   0.        ,
          0.        ,   0.        ,   0.        ],
       [ 14.        ,  99.        ,   0.        ,   0.        ,
          0.        ,   1.        ,   0.        ],
       [  5.        ,  98.        ,   0.        ,   0.        ,
          0.        ,   1.        ,   1.        ],
       [ 64.        , 102.        ,   1.        ,   0.        ,
          0.        ,   0.        ,   0.        ],
       [ 38.        , 101.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ],
       [ 49.        , 101.        ,   0.        ,   1.        ,
          0.        ,   0.        ,   0.        ],
       [ 33.        , 102.        ,   0.        ,   1.        ,
          0.    

In [19]:
X_train_tranformed.shape

(80, 7)

# 2) Mentos Zindagi ( Using Column tranformer)

In [20]:
from sklearn.compose import ColumnTransformer

In [ ]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough') # There might be some column in which we do not want to apply the transformation or want to drop that column than the remainder parameter is used 
# if we want to drop any column then we set remainder as remainder='drop' or if we want to not apply any tranformation and we want that columns then we can set remainder to remainder='passthrough

In [ ]:
# transformer = ColumnTransformer(transformers=[
#     ('tnf1',SimpleImputer(),['fever']),
#     ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
#     ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
# ],remainder='passthrough')

In [31]:
transformer.fit_transform(X_train)

TypeError: Cannot clone object. You should provide an instance of scikit-learn estimator instead of a class.